In [ ]:
import os
# import json
import itertools

import dask.dataframe as dd
import geopandas as gpd
import numpy as np
import pandas as pd

In [ ]:
import os
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.patches import Patch
import seaborn as sns

### Setup

In [ ]:
# Order of all possible bins for plotting
order = [
    '(10, 20]',
         '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 380]']

# Cutpoints
bins = [
    10,
        20, 30, 40, 50, 60, 70, 80, 90, 100, 150, 200, 380]

In [ ]:
# data_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/data/'
output_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_output/PIXC_vD_wse_2026_06_01_rel_abs/'
save_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/'
fig_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results_figures/'

### Functions

In [ ]:
def get_files(filepath):
    files = []
    for file in os.listdir(filepath):
        if not file.startswith('.'):
            # if not 'C2' in file:
            files.append(os.path.join(filepath, file))
    return files

In [ ]:
def parse_stats(df, order, num_pixels, diff_thresh, bias_thresh, corr_thresh, save_dir):
    
    # Apply quality mask
    df = df[df['n_pixels'] >= num_pixels].copy()
    df = df.groupby('site_id').filter(lambda x: len(x) >= 20)

    # Calculate various error terms
    groups = df.groupby('site_id')
    
    df = df.assign(
        wse_gauge_norm = df['wse_gauge'] - groups['wse_gauge'].transform('median'),
        wse_swot_norm  = df['wse_swot'] - groups['wse_swot'].transform('median')
    )
    
    df['wse_bias'] = df['wse_swot'] - df['wse_gauge']
    df['wse_diff'] = df['wse_swot_norm'] - df['wse_gauge_norm']
    df['wse_diff_abs'] = df['wse_diff'].abs()

    # Final filtering
    df = df[df['wse_diff_abs'] < diff_thresh].copy()
    useable = df.groupby('Bin', observed=False)['site_id'].nunique().reset_index(name='count_useable')

    # Calculate bias
    diff = df.groupby(['Bin', 'site_id'], observed=False)['wse_diff_abs'].median()
    diff = diff.reset_index()
    diff.columns = ['Bin', 'site_id', 'median_wse_diff_abs']
    
    heights = diff[diff['median_wse_diff_abs'] < bias_thresh].groupby('Bin', observed=False)['site_id'].nunique().reset_index(name='count_heights')

    # Calculate and threshold correlation
    corr = df.groupby(['Bin', 'site_id'], observed=False)[['wse_gauge_norm', 'wse_swot_norm']].corr()
    corr = corr.unstack().iloc[:, 1].reset_index()
    corr.columns = ['Bin', 'site_id', 'correlation']

    stats = corr.merge(diff, on=['site_id', 'Bin'])
    high_qual = stats[(stats['correlation'] > corr_thresh) & (np.abs(stats['median_wse_diff_abs']) < bias_thresh)]
    high_qual_ids = high_qual.site_id.to_list()
    high_qual = high_qual.groupby('Bin', observed=False)['site_id'].nunique().reset_index(name='count_high_qual')

    df_qual = unfiltered.merge(useable, on='Bin')
    df_qual = df_qual.merge(heights, on='Bin')
    df_qual = df_qual.merge(high_qual, on='Bin')

    df_qual['Bin'] = pd.Categorical(df_qual['Bin'], categories=order, ordered=True)
    df_qual = df_qual.sort_values('Bin')

    bias_label = str(bias_thresh).split('.')[1]
    if len(bias_label) < 2:
        bias_label += '0'
    corr_label = str(corr_thresh).replace('.', '')
    
    # Save with filename convention *quality_numpixels_diffthresh_biasthresh_corrthresh.parquet
    df_qual.to_parquet(
        os.path.join(
            save_dir,
            f'bennitt_wse_quality_{num_pixels}_{diff_thresh}_{bias_label}_{corr_label}.parquet'
        )
    )

## Analyze

In [ ]:
# Get list of files
files = get_files(output_dir)
print(f'Number of total files: {len(files)}')

# Read in all files
ddf = dd.read_parquet(files)

# Drop gauges sited on rivers < 10 m wide
ddf = ddf[ddf['WidthM'] >= 10]

# # Drop sites with no gauge data (should this be at least 20 obs???) # HERE
# ddf = ddf[~ddf['wse_gauge'].isna()]

# Bin reaches by width for pre-filtering analysis
ddf['Bin'] = ddf['WidthM'].map_partitions(pd.cut, bins=bins).astype(str)
ddf = ddf.reset_index(drop=True)

# Count number of reaches before filtering
unfiltered_task = ddf.groupby('Bin', observed=False)['site_id'].nunique()
unfiltered = unfiltered_task.compute().reset_index(name='count_unfil')

unfiltered['Bin'] = pd.Categorical(unfiltered['Bin'], categories=order, ordered=True)
unfiltered = unfiltered.sort_values('Bin')

# Filter and sort data (for Hampel filter, which we are not currently using)
ddf = ddf[
    ddf['wse_gauge'].notnull() & 
    ddf['wse_swot'].notnull()
    ][[
        'date', 'n_pixels', 'wse_gauge',
        'wse_swot', 'site_id', 'WidthM',
        'Bin'
    ]]

df = ddf.compute().sort_values(['site_id', 'date'])
print(f'Number of valid files: {len(df['site_id'].unique())}')

In [ ]:
unfiltered['count_unfil'].sum()

### ONE OF THESE REACHES HAS NO GAUGE DATA

In [ ]:
num_pixels = [3, 5, 7, 10]
diff_thresh = [3, 5, 7, 10]
bias_thresh = [0.18, 0.20, 0.25, 0.30, 0.40, 0.50]
corr_thresh = [0.5, 0.6, 0.7, 0.8, 0.9]

In [ ]:
# Generate results altering number of pixels and outliering
for n, d in itertools.product(num_pixels, diff_thresh):
    parse_stats(df=df, 
                order=order,
                num_pixels=n,
                diff_thresh=d,
                bias_thresh=0.3,
                corr_thresh=0.6,
                save_dir=save_dir)

In [ ]:
# Generate results altering bias and correlation
for b, c in itertools.product(bias_thresh, corr_thresh):
    parse_stats(df=df, 
                order=order,
                num_pixels=5,
                diff_thresh=5,
                bias_thresh=b,
                corr_thresh=c,
                save_dir=save_dir)

In [ ]:
# test = pd.read_parquet('/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/bennitt_wse_quality_10_5_30_06.parquet')

In [ ]:
# # 3. Add text labels (count_unfil) at the top of each panel
# counts_map = df_current.set_index('Bin')['count_unfil'].to_dict()
# y_pos = 102 

# for i, bin_name in enumerate(order):
#     if bin_name in counts_map:
#         count_val = counts_map[bin_name]
#         if not pd.isna(count_val):
#             ax.text(i, y_pos, f'{int(count_val):,}', 
#                     ha='center', va='bottom', fontsize=9, # Smaller text for grid layout
#                     fontweight='bold', color='dimgray')

In [ ]:
# --- Setup Lists and Styling ---
num_pixels = [3, 5, 7, 10]
diff_thresh = [3, 5, 7, 10]
bias_thresh = 0.3
corr_thresh = 0.6

palette = [
    'silver',
    sns.color_palette('tab20b')[2],
    '#8DAE91',
    sns.color_palette('tab20b')[-3]
]

# Initialize 5x5 subplots grid with shared axes
fig, axes = plt.subplots(4, 4, figsize=(24, 20), sharex=True, sharey=True)

# Loop through rows (num pixels) and columns (diff)
for row_idx, n in enumerate(num_pixels):
    for col_idx, d in enumerate(diff_thresh):
        ax = axes[row_idx, col_idx]
        
        # 1. Recreate your exact filename logic to read the file
        bias_label = str(bias_thresh).split('.')[1]
        if len(bias_label) < 2:
            bias_label += '0'
        corr_label = str(corr_thresh).replace('.', '')
        
        filename = f'bennitt_wse_quality_{n}_{d}_{bias_label}_{corr_label}.parquet'
        file_path = os.path.join(save_dir, filename)
        
        if not os.path.exists(file_path):
            ax.text(0.5, 0.5, 'Missing File', ha='center', va='center', color='red', fontsize=12)
            continue
            
        df_current = pd.read_parquet(file_path)
        if df_current.empty:
            ax.text(0.5, 0.5, 'Empty Data', ha='center', va='center', color='orange', fontsize=12)
            continue

        df_current['total'] = 100
        df_current['useable'] = ((df_current['count_useable'] / df_current['count_unfil']) * 100).round(2)
        df_current['heights'] = ((df_current['count_heights'] / df_current['count_unfil']) * 100).round(2)
        df_current['quality'] = ((df_current['count_high_qual'] / df_current['count_unfil']) * 100).round(2)

        # 2. Render overlapping bar plots on the current subplot axis
        sns.barplot(ax=ax, data=df_current, x='Bin', y='total', color=palette[0], alpha=0.7, order=order)
        sns.barplot(ax=ax, data=df_current, x='Bin', y='useable', color=palette[1], alpha=0.7, order=order)
        sns.barplot(ax=ax, data=df_current, x='Bin', y='heights', color=palette[2], alpha=0.7, order=order)
        sns.barplot(ax=ax, data=df_current, x='Bin', y='quality', color=palette[3], order=order)

        # Create a mapping of Bin to the 'quality' value
        quality_map = df_current.set_index('Bin')['quality'].to_dict()
        
        # Get values safely, defaulting to 0 or NaN if the bin string isn't found
        q_30_40 = quality_map.get('(30, 40]', 0)
        q_40_50 = quality_map.get('(40, 50]', 0)
        
        # Format strings (handling floats or formatting as integers if preferred)
        textstr = f"30-40m: {q_30_40:.1f}%\n40-50m: {q_40_50:.1f}%"
        
        # Place text using transform=ax.transAxes so (0,1) is exactly the top left corner of the subplot bounding box
        ax.text(0.97, 0.03, textstr, transform=ax.transAxes, 
                fontsize=18,
                fontweight='semibold',
                color='k',
                # palette[3],
                va='bottom', ha='right',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          alpha=0.9, edgecolor='none'))
        
        # ax.set_ylim(0, 115)
        ax.set_ylim(0, 100)
        ax.yaxis.grid(True)
        ax.set_axisbelow(True)
        ax.set_xlabel('') # Clear individual labels to avoid clutter
        ax.set_ylabel('')

        # 5. Label outer matrix headers (Top columns and Left rows)
        if row_idx == 0:
            ax.set_title(f'{d} m', fontsize=18, fontweight='bold', pad=15)
        if col_idx == 0:
            # Overwritten below by global supylabel, but kept for row-specific details if needed
            ax.set_ylabel(f'{n}', fontsize=18, fontweight='bold', labelpad=15)

for ax in axes[-1, :]:
    ax.tick_params(axis='x', rotation=-45, labelsize=16)
    for label in ax.get_xticklabels():
        label.set_fontweight('semibold')

# Format shared Y-axis tick marks for all leftmost subplots
for ax in axes[:, 0]:
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
    ax.tick_params(axis='y', labelsize=16)
    for label in ax.get_yticklabels():
        label.set_fontweight('semibold')

# --- Global Axis Labels ---
# These apply cleanly to the entire 5x5 figure matrix boundary
fig.supxlabel("River width (m)", fontsize=20, fontweight='bold', y=0.04)
fig.supylabel("Percentage of reaches (%)", fontsize=20, fontweight='bold', x=0.01)

# --- Shared Global Legend ---
legend_labels = ['Insufficient data', 'Unacceptable bias', 'Unreasonable timeseries', 'Sensible SWOT WSE']
handles = [Patch(facecolor=color, label=label) for color, label in zip(palette, legend_labels)]

# Placing the legend beneath the global X label to prevent overlaps
fig.legend(handles, legend_labels, loc='lower center', ncol=4, 
           framealpha=0.95, prop={'size': 20, 'weight': 'semibold'},
           bbox_to_anchor=(0.5, 0)
          )

# Optional: Tight layout adjustments to make sure global labels fit nicely outside the axes
plt.tight_layout(rect=[0.02, 0.05, 1, 1])

plt.savefig(os.path.join(fig_dir, 'bennitt_wse_quality_matrix_4x4.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- Setup Lists and Styling ---
bias_thresh = [0.18, 0.20, 0.25, 0.30, 0.40, 0.50]
corr_thresh = [0.5, 0.6, 0.7, 0.8, 0.9]
num_pixels = 5
diff_thresh = 5

palette = [
    'silver',
    sns.color_palette('tab20b')[2],
    '#8DAE91',
    sns.color_palette('tab20b')[-3]
]

sns.set_style('whitegrid')

# Initialize 5x5 subplots grid with shared axes
fig, axes = plt.subplots(6, 5, figsize=(24, 28), sharex=True, sharey=True)

# Loop through rows (Bias) and columns (Corr)
for row_idx, b in enumerate(bias_thresh):
    for col_idx, c in enumerate(corr_thresh):
        ax = axes[row_idx, col_idx]
        
        # 1. Recreate your exact filename logic to read the file
        bias_label = str(b).split('.')[1]
        if len(bias_label) < 2:
            bias_label += '0'
        corr_label = str(c).replace('.', '')
        
        filename = f'bennitt_wse_quality_{num_pixels}_{diff_thresh}_{bias_label}_{corr_label}.parquet'
        file_path = os.path.join(save_dir, filename)
        
        if not os.path.exists(file_path):
            ax.text(0.5, 0.5, 'Missing File', ha='center', va='center', color='red', fontsize=12)
            continue
            
        df_current = pd.read_parquet(file_path)
        if df_current.empty:
            ax.text(0.5, 0.5, 'Empty Data', ha='center', va='center', color='orange', fontsize=12)
            continue

        df_current['total'] = 100
        df_current['useable'] = ((df_current['count_useable'] / df_current['count_unfil']) * 100).round(2)
        df_current['heights'] = ((df_current['count_heights'] / df_current['count_unfil']) * 100).round(2)
        df_current['quality'] = ((df_current['count_high_qual'] / df_current['count_unfil']) * 100).round(2)

        # 2. Render overlapping bar plots on the current subplot axis
        sns.barplot(ax=ax, data=df_current, x='Bin', y='total', color=palette[0], alpha=0.7, order=order)
        sns.barplot(ax=ax, data=df_current, x='Bin', y='useable', color=palette[1], alpha=0.7, order=order)
        sns.barplot(ax=ax, data=df_current, x='Bin', y='heights', color=palette[2], alpha=0.7, order=order)
        sns.barplot(ax=ax, data=df_current, x='Bin', y='quality', color=palette[3], order=order)

        # Create a mapping of Bin to the 'quality' value
        quality_map = df_current.set_index('Bin')['quality'].to_dict()
        
        # Get values safely, defaulting to 0 or NaN if the bin string isn't found
        q_30_40 = quality_map.get('(30, 40]', 0)
        q_40_50 = quality_map.get('(40, 50]', 0)
        
        # Format strings (handling floats or formatting as integers if preferred)
        textstr = f"30-40m: {q_30_40:.1f}%\n40-50m: {q_40_50:.1f}%"
        
        # Place text using transform=ax.transAxes so (0,1) is exactly the top left corner of the subplot bounding box
        ax.text(0.97, 0.03, textstr, transform=ax.transAxes, 
                fontsize=18,
                fontweight='semibold',
                color='k',
                # palette[3],
                va='bottom', ha='right',
                bbox=dict(boxstyle='round,pad=0.2',
                          facecolor='white', alpha=0.9, edgecolor='none'))
        
        # ax.set_ylim(0, 115)
        ax.set_ylim(0, 100)
        ax.yaxis.grid(True)
        ax.set_axisbelow(True)
        ax.set_xlabel('') # Clear individual labels to avoid clutter
        ax.set_ylabel('')

        # 5. Label outer matrix headers (Top columns and Left rows)
        if row_idx == 0:
            ax.set_title(f'{c}', fontsize=18, fontweight='bold', pad=15)
        if col_idx == 0:
            # Overwritten below by global supylabel, but kept for row-specific details if needed
            ax.set_ylabel(f'{b:.2f} m', fontsize=18, fontweight='bold', labelpad=15)

for ax in axes[-1, :]:
    ax.tick_params(axis='x', rotation=-45, labelsize=14)
    for label in ax.get_xticklabels():
        label.set_fontweight('semibold')

# Format shared Y-axis tick marks for all leftmost subplots
for ax in axes[:, 0]:
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
    ax.tick_params(axis='y', labelsize=16)
    for label in ax.get_yticklabels():
        label.set_fontweight('semibold')

# --- Global Axis Labels ---
# These apply cleanly to the entire 5x5 figure matrix boundary
fig.supxlabel("River width (m)", fontsize=20, fontweight='bold', y=0.04)
fig.supylabel("Percentage of reaches (%)", fontsize=20, fontweight='bold', x=0.01)

# --- Shared Global Legend ---
legend_labels = ['Insufficient data', 'Unacceptable bias',
                 'Unreasonable timeseries', 'Sensible SWOT WSE']
handles = [Patch(facecolor=color, label=label) for color, label in zip(palette, legend_labels)]

# Placing the legend beneath the global X label to prevent overlaps
fig.legend(handles, legend_labels, loc='lower center', ncol=4, 
           framealpha=0.95,
           prop={'size': 20, 'weight': 'semibold'},
           bbox_to_anchor=(0.5, 0)
          )

# Optional: Tight layout adjustments to make sure global labels fit nicely outside the axes
plt.tight_layout(rect=[0.02, 0.05, 1, 1])

plt.savefig(os.path.join(fig_dir, 'bennitt_wse_quality_matrix_6x5.png'), dpi=300, bbox_inches='tight')
plt.show()